# 🛒 Вариант 10 — Подсчёт товаров на полке магазина
**Практика: Системы искусственного интеллекта | МТУСИ БВТ22**

| # | Модель | Семейство |
|---|--------|-----------|
| 1 | YOLOv8n | One-stage anchor-free |
| 2 | YOLOv5s | One-stage anchor-based |
| 3 | Faster R-CNN ResNet-50 FPN | Two-stage |
| 4 | SSDLite MobileNetV3 | Lightweight one-stage |
| 5 | RT-DETR-l | Transformer-based (без NMS) |

Запускай ячейки **по порядку** сверху вниз.

---
## ✅ Шаг 0. Проверка GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA доступна:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Память:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
else:
    print('GPU не найден! Среда выполнения -> Сменить тип среды -> T4 GPU')

---
## 📦 Шаг 1. Установка зависимостей

In [ ]:
!pip install -q ultralytics roboflow gradio matplotlib pillow pyyaml
!git clone -q https://github.com/ultralytics/yolov5.git 2>/dev/null || echo 'YOLOv5 уже скачан'
!pip install -q -r yolov5/requirements.txt
print('Все зависимости установлены')

---
## 📂 Шаг 2. Скачать датасет SKU-110K

**Получи бесплатный API-ключ Roboflow:**
1. Зайди на [app.roboflow.com](https://app.roboflow.com)
2. Зарегистрируйся (бесплатно)
3. Settings -> API Keys -> скопируй ключ
4. Вставь в поле ниже

In [ ]:
import os, json, yaml, time, csv, datetime
from pathlib import Path
import numpy as np
from PIL import Image

ROBOFLOW_API_KEY = 'ТВОЙ_КЛЮЧ_ЗДЕСЬ'  # <-- вставь сюда

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED   = 42
DATA_DIR   = Path('./data')
DATA_VOC   = Path('./data_voc')
RUNS_DIR   = Path('./runs')
MODELS_DIR = Path('./models')
for d in [DATA_DIR, DATA_VOC, RUNS_DIR, MODELS_DIR]:
    d.mkdir(exist_ok=True)
print('Устройство:', DEVICE)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('sku-detection').project('sku-110k-subset')
dataset = project.version(1).download('yolov8', location=str(DATA_DIR))
print('Датасет скачан в', DATA_DIR)

In [ ]:
# Статистика датасета
print('Статистика датасета:')
print(f'{"Split":<10} {"Images":<10} {"Objects":<10} {"Avg/img"}')
print('-' * 42)
for split in ['train', 'valid', 'test']:
    img_dir = DATA_DIR / split / 'images'
    lbl_dir = DATA_DIR / split / 'labels'
    if not img_dir.exists(): continue
    imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    lbls = list(lbl_dir.glob('*.txt')) if lbl_dir.exists() else []
    objs = sum(len(open(l).readlines()) for l in lbls)
    print(f'{split:<10} {len(imgs):<10} {objs:<10} {round(objs/max(len(imgs),1),1)}')

# data.yaml
data_yaml_path = str((DATA_DIR / 'data.yaml').resolve())
cfg = {'path': str(DATA_DIR.resolve()), 'train': 'train/images',
       'val': 'valid/images', 'test': 'test/images', 'nc': 1, 'names': ['product']}
with open(data_yaml_path, 'w') as f:
    yaml.dump(cfg, f)
print('data.yaml создан')

In [ ]:
# Конвертация в Pascal VOC для Faster R-CNN и SSD
def yolo_to_voc(data_dir, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    paths = {}
    for split in ['train', 'valid', 'test']:
        img_dir = Path(data_dir) / split / 'images'
        lbl_dir = Path(data_dir) / split / 'labels'
        if not img_dir.exists(): continue
        anns = []
        for img_path in sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))):
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            try: W, H = Image.open(img_path).size
            except: W, H = 640, 640
            boxes = []
            if lbl_path.exists():
                for line in open(lbl_path):
                    p = line.strip().split()
                    if len(p) < 5: continue
                    cls, cx, cy, w, h = int(p[0]), *map(float, p[1:5])
                    x1,y1 = (cx-w/2)*W, (cy-h/2)*H
                    x2,y2 = (cx+w/2)*W, (cy+h/2)*H
                    if x2>x1 and y2>y1:
                        boxes.append({'label': cls, 'bbox': [x1,y1,x2,y2]})
            anns.append({'image_path': str(img_path.resolve()), 'width': W, 'height': H, 'boxes': boxes})
        out = output_dir / f'{split}.json'
        json.dump(anns, open(out,'w'), indent=2)
        paths[split] = str(out)
        print(f'{split}: {len(anns)} изображений')
    return paths

print('Конвертация в Pascal VOC...')
voc_paths = yolo_to_voc(DATA_DIR, DATA_VOC)
print('Готово')

---
## Вспомогательные классы (Dataset, метрики)

In [ ]:
import torch
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import box_iou

class ShelfDataset(Dataset):
    def __init__(self, json_path, transforms=None):
        self.data = json.load(open(json_path))
        self.transforms = transforms
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(item['image_path']).convert('RGB')
        boxes, labels = [], []
        for ann in item['boxes']:
            x1,y1,x2,y2 = ann['bbox']
            if x2>x1 and y2>y1:
                boxes.append([x1,y1,x2,y2])
                labels.append(ann['label']+1)
        boxes  = torch.as_tensor(boxes,  dtype=torch.float32) if boxes  else torch.zeros((0,4))
        labels = torch.as_tensor(labels, dtype=torch.int64)   if labels else torch.zeros((0,), dtype=torch.int64)
        target = {'boxes': boxes, 'labels': labels, 'image_id': torch.tensor([idx])}
        img = self.transforms(image) if self.transforms else T.ToTensor()(image)
        return img, target

def get_transforms(train=True):
    tfms = [T.ToTensor()]
    if train: tfms += [T.RandomHorizontalFlip(0.5), T.ColorJitter(brightness=0.3, contrast=0.3)]
    tfms.append(T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]))
    return T.Compose(tfms)

def collate_fn(batch): return tuple(zip(*batch))

@torch.no_grad()
def evaluate_map(model, loader, device, iou_thr=0.5, conf_thr=0.5):
    model.eval()
    tp, fp, fn = 0, 0, 0
    for images, targets in loader:
        images = [img.to(device) for img in images]
        outputs = model(images)
        for out, tgt in zip(outputs, targets):
            pb = out['boxes'].cpu()
            ps = out['scores'].cpu()
            gb = tgt['boxes'].cpu()
            pb = pb[ps > conf_thr]
            if len(gb)==0 and len(pb)==0: continue
            if len(gb)==0: fp += len(pb); continue
            if len(pb)==0: fn += len(gb); continue
            iou = box_iou(pb, gb)
            matched = set()
            for pi in range(len(pb)):
                best_iou, best_gt = iou[pi].max(0)
                if best_iou >= iou_thr and best_gt.item() not in matched:
                    tp += 1; matched.add(best_gt.item())
                else: fp += 1
            fn += len(gb) - len(matched)
    prec = tp/max(tp+fp,1)
    rec  = tp/max(tp+fn,1)
    f1   = 2*prec*rec/max(prec+rec,1e-6)
    return {'map50': round(f1,4), 'precision': round(prec,4), 'recall': round(rec,4)}

ALL_METRICS = []
print('Вспомогательные функции загружены')

---
## Модель 1/5 — YOLOv8n (anchor-free, one-stage)
Современный быстрый детектор без якорей.

In [ ]:
from ultralytics import YOLO
EPOCHS_YOLO = 50  # уменьши до 10 для быстрого теста

print('Обучение 1/5: YOLOv8n — anchor-free, one-stage')
model_v8 = YOLO('yolov8n.pt')
t0 = time.time()
model_v8.train(
    data=data_yaml_path, epochs=EPOCHS_YOLO, batch=16, imgsz=640,
    device=DEVICE, project='./runs/yolov8', name='train', seed=SEED,
    patience=15, lr0=0.01, mosaic=1.0, plots=True, exist_ok=True,
)
train_min_v8 = round((time.time()-t0)/60, 1)

best_v8 = './runs/yolov8/train/weights/best.pt'
m = YOLO(best_v8).val(data=data_yaml_path, split='test', verbose=False)

mv8 = YOLO(best_v8)
dummy = torch.zeros(1,3,640,640)
[mv8.predict(dummy, verbose=False) for _ in range(10)]
t1 = time.time()
[mv8.predict(dummy, verbose=False) for _ in range(100)]
inf_ms_v8 = round((time.time()-t1)/100*1000, 2)
size_v8 = round(Path(best_v8).stat().st_size/1024/1024, 1)

res_v8 = {
    'Модель': 'YOLOv8n', 'Семейство': 'One-stage anchor-free', 'Input': '640px',
    'Эпохи': EPOCHS_YOLO, 'mAP@0.5': round(float(m.box.map50),4),
    'Precision': round(float(m.box.mp),4), 'Recall': round(float(m.box.mr),4),
    'мс/кадр': inf_ms_v8, 'МБ': size_v8, 'Обучение_мин': train_min_v8,
    'weights': best_v8, 'notes': 'Быстрый anchor-free, хорошо на плотных сценах'
}
ALL_METRICS.append(res_v8)
print(f"YOLOv8n готов | mAP@0.5={res_v8['mAP@0.5']} | {inf_ms_v8}мс | {size_v8}МБ")

---
## Модель 2/5 — YOLOv5s (anchor-based, one-stage)
Классическое якорное семейство YOLO.

In [ ]:
import subprocess, sys, csv as csv_mod
EPOCHS_V5 = 50
print('Обучение 2/5: YOLOv5s — anchor-based, one-stage')
t0 = time.time()
subprocess.run([
    sys.executable, 'yolov5/train.py',
    '--data', data_yaml_path, '--weights', 'yolov5s.pt',
    '--epochs', str(EPOCHS_V5), '--batch-size', '16', '--imgsz', '640',
    '--device', DEVICE, '--project', './runs/yolov5', '--name', 'train',
    '--seed', str(SEED), '--patience', '15', '--exist-ok',
], check=True)
train_min_v5 = round((time.time()-t0)/60, 1)

best_v5 = './runs/yolov5/train/weights/best.pt'
map50_v5 = prec_v5 = rec_v5 = 0.0
rcsv = './runs/yolov5/train/results.csv'
if Path(rcsv).exists():
    rows = list(csv_mod.DictReader(open(rcsv)))
    if rows:
        last = rows[-1]
        try:
            prec_v5  = float(last.get('   metrics/precision', 0))
            rec_v5   = float(last.get('      metrics/recall', 0))
            map50_v5 = float(last.get('     metrics/mAP_0.5', 0))
        except: pass

sys.path.insert(0, 'yolov5')
from models.experimental import attempt_load
from utils.torch_utils import select_device as yv5_dev
dev5 = yv5_dev(DEVICE)
mv5  = attempt_load(best_v5, device=dev5)
mv5.eval()
d5 = torch.zeros(1,3,640,640).to(dev5)
[mv5(d5) for _ in range(10)]
t1 = time.time()
with torch.no_grad(): [mv5(d5) for _ in range(100)]
inf_ms_v5 = round((time.time()-t1)/100*1000, 2)
size_v5 = round(Path(best_v5).stat().st_size/1024/1024, 1)

res_v5 = {
    'Модель': 'YOLOv5s', 'Семейство': 'One-stage anchor-based', 'Input': '640px',
    'Эпохи': EPOCHS_V5, 'mAP@0.5': round(map50_v5,4),
    'Precision': round(prec_v5,4), 'Recall': round(rec_v5,4),
    'мс/кадр': inf_ms_v5, 'МБ': size_v5, 'Обучение_мин': train_min_v5,
    'weights': best_v5, 'notes': 'Классический anchor-based baseline'
}
ALL_METRICS.append(res_v5)
print(f"YOLOv5s готов | mAP@0.5={res_v5['mAP@0.5']} | {inf_ms_v5}мс | {size_v5}МБ")

---
## Модель 3/5 — Faster R-CNN ResNet-50 FPN (two-stage)
Двухстадийный детектор: RPN + ROI Head.

In [ ]:
import torch.nn as nn
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

EPOCHS_FRCNN = 30
BATCH_FRCNN  = 4
print('Обучение 3/5: Faster R-CNN ResNet-50 FPN — two-stage')

def build_frcnn():
    m = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
    in_f = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(in_f, 2)
    return m

torch.manual_seed(SEED)
device = torch.device(DEVICE)
train_ld = DataLoader(ShelfDataset(str(DATA_VOC/'train.json'), get_transforms(True)),  BATCH_FRCNN, shuffle=True,  num_workers=2, collate_fn=collate_fn)
val_ld   = DataLoader(ShelfDataset(str(DATA_VOC/'valid.json'), get_transforms(False)), BATCH_FRCNN, shuffle=False, num_workers=2, collate_fn=collate_fn)
test_ld  = DataLoader(ShelfDataset(str(DATA_VOC/'test.json'),  get_transforms(False)), BATCH_FRCNN, shuffle=False, num_workers=2, collate_fn=collate_fn)

frcnn = build_frcnn().to(device)
opt = torch.optim.SGD(
    [{'params':[p for n,p in frcnn.named_parameters() if 'backbone' in n], 'lr':5e-4},
     {'params':[p for n,p in frcnn.named_parameters() if 'backbone' not in n], 'lr':5e-3}],
    momentum=0.9, weight_decay=5e-4)
sched = torch.optim.lr_scheduler.MultiStepLR(opt, [int(EPOCHS_FRCNN*0.6), int(EPOCHS_FRCNN*0.8)], gamma=0.1)

best_map_f = 0.0
Path('./runs/fasterrcnn').mkdir(parents=True, exist_ok=True)
best_path_f = Path('./runs/fasterrcnn/best.pth')
t0 = time.time()

for epoch in range(1, EPOCHS_FRCNN+1):
    frcnn.train(); total_loss = 0
    for imgs, tgts in train_ld:
        imgs = [x.to(device) for x in imgs]
        tgts = [{k:v.to(device) for k,v in t.items()} for t in tgts]
        if any(len(t['boxes'])==0 for t in tgts): continue
        losses = sum(frcnn(imgs,tgts).values())
        opt.zero_grad(); losses.backward()
        torch.nn.utils.clip_grad_norm_(frcnn.parameters(), 1.0)
        opt.step(); total_loss += losses.item()
    sched.step()
    if epoch % 5 == 0 or epoch == EPOCHS_FRCNN:
        vm = evaluate_map(frcnn, val_ld, device)
        print(f'Epoch {epoch} loss={total_loss/len(train_ld):.3f} mAP={vm["map50"]} P={vm["precision"]} R={vm["recall"]}')
        if vm['map50'] > best_map_f:
            best_map_f = vm['map50']; torch.save(frcnn.state_dict(), best_path_f)

train_min_f = round((time.time()-t0)/60, 1)
frcnn.load_state_dict(torch.load(best_path_f, map_location=device))
tm_f = evaluate_map(frcnn, test_ld, device)
frcnn.eval()
d = [torch.zeros(3,640,640).to(device)]
with torch.no_grad():
    [frcnn(d) for _ in range(10)]
    t1 = time.time()
    [frcnn(d) for _ in range(100)]
inf_ms_f = round((time.time()-t1)/100*1000, 2)
size_f = round(best_path_f.stat().st_size/1024/1024, 1)

res_frcnn = {
    'Модель': 'Faster R-CNN ResNet-50 FPN', 'Семейство': 'Two-stage (RPN+ROI)', 'Input': '640px',
    'Эпохи': EPOCHS_FRCNN, 'mAP@0.5': tm_f['map50'],
    'Precision': tm_f['precision'], 'Recall': tm_f['recall'],
    'мс/кадр': inf_ms_f, 'МБ': size_f, 'Обучение_мин': train_min_f,
    'weights': str(best_path_f), 'notes': 'Высокая точность на мелких объектах, медленнее YOLO'
}
ALL_METRICS.append(res_frcnn)
print(f"Faster R-CNN готов | mAP={tm_f['map50']} | {inf_ms_f}мс | {size_f}МБ")

---
## Модель 4/5 — SSDLite MobileNetV3 (lightweight one-stage)
Самый лёгкий детектор, оптимизирован для мобильных устройств.

In [ ]:
from functools import partial
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead

EPOCHS_SSD = 40; BATCH_SSD = 16; IMG_SSD = 320
print('Обучение 4/5: SSDLite MobileNetV3 — lightweight one-stage')

def ssd_tfm(train=True):
    t = [T.Resize((IMG_SSD,IMG_SSD)), T.ToTensor()]
    if train: t.insert(1, T.RandomHorizontalFlip(0.5))
    t.append(T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]))
    return T.Compose(t)

ssd_tl = DataLoader(ShelfDataset(str(DATA_VOC/'train.json'), ssd_tfm(True)),  BATCH_SSD, shuffle=True,  num_workers=2, collate_fn=collate_fn)
ssd_vl = DataLoader(ShelfDataset(str(DATA_VOC/'valid.json'), ssd_tfm(False)), BATCH_SSD, shuffle=False, num_workers=2, collate_fn=collate_fn)
ssd_tsl= DataLoader(ShelfDataset(str(DATA_VOC/'test.json'),  ssd_tfm(False)), BATCH_SSD, shuffle=False, num_workers=2, collate_fn=collate_fn)

ssd_model = ssdlite320_mobilenet_v3_large(weights=SSDLite320_MobileNet_V3_Large_Weights.DEFAULT)
in_ch  = [672,480,512,256,256,128]
n_anch = ssd_model.anchor_generator.num_anchors_per_location()
ssd_model.head.classification_head = SSDLiteClassificationHead(
    in_channels=in_ch, num_anchors=n_anch, num_classes=2,
    norm_layer=partial(nn.BatchNorm2d, eps=0.001, momentum=0.03))
ssd_model.to(device)

ssd_opt = torch.optim.SGD(
    [{'params':[p for n,p in ssd_model.named_parameters() if 'backbone' in n], 'lr':1e-3},
     {'params':[p for n,p in ssd_model.named_parameters() if 'backbone' not in n], 'lr':1e-2}],
    momentum=0.9, weight_decay=1e-4)
ssd_sched = torch.optim.lr_scheduler.CosineAnnealingLR(ssd_opt, T_max=EPOCHS_SSD)
best_map_s = 0.0
Path('./runs/ssd').mkdir(parents=True, exist_ok=True)
best_path_s = Path('./runs/ssd/best.pth')
t0 = time.time()

for epoch in range(1, EPOCHS_SSD+1):
    ssd_model.train(); total_loss = 0
    for imgs, tgts in ssd_tl:
        imgs = [x.to(device) for x in imgs]
        tgts = [{k:v.to(device) for k,v in t.items()} for t in tgts]
        if any(len(t['boxes'])==0 for t in tgts): continue
        losses = sum(ssd_model(imgs,tgts).values())
        ssd_opt.zero_grad(); losses.backward()
        torch.nn.utils.clip_grad_norm_(ssd_model.parameters(), 5.0)
        ssd_opt.step(); total_loss += losses.item()
    ssd_sched.step()
    if epoch % 5 == 0 or epoch == EPOCHS_SSD:
        vm = evaluate_map(ssd_model, ssd_vl, device)
        print(f'Epoch {epoch} loss={total_loss/len(ssd_tl):.3f} mAP={vm["map50"]}')
        if vm['map50'] > best_map_s:
            best_map_s = vm['map50']; torch.save(ssd_model.state_dict(), best_path_s)

train_min_s = round((time.time()-t0)/60, 1)
ssd_model.load_state_dict(torch.load(best_path_s, map_location=device))
tm_s = evaluate_map(ssd_model, ssd_tsl, device)
ssd_model.eval()
d = [torch.zeros(3,IMG_SSD,IMG_SSD).to(device)]
with torch.no_grad():
    [ssd_model(d) for _ in range(10)]
    t1 = time.time()
    [ssd_model(d) for _ in range(100)]
inf_ms_s = round((time.time()-t1)/100*1000, 2)
size_s = round(best_path_s.stat().st_size/1024/1024, 1)

res_ssd = {
    'Модель': 'SSDLite MobileNetV3', 'Семейство': 'Lightweight one-stage SSD', 'Input': f'{IMG_SSD}px',
    'Эпохи': EPOCHS_SSD, 'mAP@0.5': tm_s['map50'],
    'Precision': tm_s['precision'], 'Recall': tm_s['recall'],
    'мс/кадр': inf_ms_s, 'МБ': size_s, 'Обучение_мин': train_min_s,
    'weights': str(best_path_s), 'notes': 'Самая лёгкая модель, компромисс качество/скорость'
}
ALL_METRICS.append(res_ssd)
print(f"SSDLite готов | mAP={tm_s['map50']} | {inf_ms_s}мс | {size_s}МБ")

---
## Модель 5/5 — RT-DETR-l (Transformer-based, без NMS)
Трансформерный детектор. Не требует NMS — attention сам устраняет дубликаты.

In [ ]:
from ultralytics import RTDETR
EPOCHS_RTDETR = 50
print('Обучение 5/5: RT-DETR-l — Transformer, без NMS')

rtdetr = RTDETR('rtdetr-l.pt')
t0 = time.time()
rtdetr.train(
    data=data_yaml_path, epochs=EPOCHS_RTDETR, batch=8, imgsz=640,
    device=DEVICE, project='./runs/rtdetr', name='train', seed=SEED,
    patience=15, lr0=0.0001, weight_decay=0.0001,
    optimizer='AdamW', warmup_epochs=2, plots=True, exist_ok=True,
)
train_min_rt = round((time.time()-t0)/60, 1)

best_rt = './runs/rtdetr/train/weights/best.pt'
m = RTDETR(best_rt).val(data=data_yaml_path, split='test', verbose=False)

mrt = RTDETR(best_rt)
dummy = torch.zeros(1,3,640,640)
[mrt.predict(dummy, verbose=False) for _ in range(10)]
t1 = time.time()
[mrt.predict(dummy, verbose=False) for _ in range(100)]
inf_ms_rt = round((time.time()-t1)/100*1000, 2)
size_rt = round(Path(best_rt).stat().st_size/1024/1024, 1)

res_rtdetr = {
    'Модель': 'RT-DETR-l', 'Семейство': 'Transformer-based (без NMS)', 'Input': '640px',
    'Эпохи': EPOCHS_RTDETR, 'mAP@0.5': round(float(m.box.map50),4),
    'Precision': round(float(m.box.mp),4), 'Recall': round(float(m.box.mr),4),
    'мс/кадр': inf_ms_rt, 'МБ': size_rt, 'Обучение_мин': train_min_rt,
    'weights': best_rt, 'notes': 'Трансформер без NMS, лучше на перекрытиях'
}
ALL_METRICS.append(res_rtdetr)
print(f"RT-DETR-l готов | mAP={res_rtdetr['mAP@0.5']} | {inf_ms_rt}мс | {size_rt}МБ")

---
## Сравнение всех 5 архитектур

In [ ]:
import matplotlib.pyplot as plt

print('=' * 95)
print('ИТОГОВАЯ ТАБЛИЦА СРАВНЕНИЯ — Вариант 10: Подсчёт товаров на полке')
print('=' * 95)
print(f'{"Модель":<28} {"Семейство":<26} {"mAP@0.5":<9} {"Precision":<11} {"Recall":<9} {"мс/кадр":<10} {"МБ"}')
print('-' * 95)
best_map = max(r['mAP@0.5'] for r in ALL_METRICS)
for r in ALL_METRICS:
    mark = ' ЛУЧШИЙ' if r['mAP@0.5'] == best_map else ''
    print(f"{r['Модель']:<28} {r['Семейство']:<26} {r['mAP@0.5']:<9} {r['Precision']:<11} {r['Recall']:<9} {r['мс/кадр']:<10} {r['МБ']}{mark}")
print('=' * 95)

Path('./runs/results').mkdir(parents=True, exist_ok=True)
json.dump({'task': 'Вариант 10', 'models': ALL_METRICS}, open('./runs/results/comparison.json','w'), indent=2, ensure_ascii=False)

import csv as csv_mod
with open('./runs/results/comparison.csv','w',newline='',encoding='utf-8-sig') as f:
    w = csv_mod.DictWriter(f, fieldnames=list(ALL_METRICS[0].keys()))
    w.writeheader(); w.writerows(ALL_METRICS)
print('Сохранено: runs/results/comparison.json и comparison.csv')

In [ ]:
# Графики сравнения
names  = [r['Модель'] for r in ALL_METRICS]
short  = [n.split()[0] for n in names]
map50  = [r['mAP@0.5'] for r in ALL_METRICS]
prec   = [r['Precision'] for r in ALL_METRICS]
rec    = [r['Recall'] for r in ALL_METRICS]
ms_v   = [r['мс/кадр'] for r in ALL_METRICS]
mb_v   = [r['МБ'] for r in ALL_METRICS]
colors = ['#2196F3','#4CAF50','#FF9800','#E91E63','#9C27B0']
x = np.arange(len(names))

fig, axes = plt.subplots(1, 3, figsize=(18,5))
fig.suptitle('Сравнение архитектур — Подсчёт товаров на полке (Вариант 10)', fontsize=13, fontweight='bold')

ax = axes[0]; w = 0.25
for bars, vals, lbl in [(ax.bar(x-w, map50, w, color='#2196F3', alpha=0.85), map50, 'mAP@0.5'),
                         (ax.bar(x,   prec,  w, color='#4CAF50', alpha=0.85), prec,  'Precision'),
                         (ax.bar(x+w, rec,   w, color='#FF9800', alpha=0.85), rec,   'Recall')]:
    bars.set_label(lbl)
    for b in bars: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{b.get_height():.3f}', ha='center', fontsize=7)
ax.set_title('Качество детекции'); ax.set_xticks(x); ax.set_xticklabels(short, rotation=20, ha='right', fontsize=8)
ax.set_ylim(0,1.1); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

ax = axes[1]
bars = ax.bar(short, ms_v, color=colors, alpha=0.85)
ax.set_title('Скорость инференса (мс/кадр)')
ax.set_xticklabels(short, rotation=20, ha='right', fontsize=8); ax.grid(axis='y', alpha=0.3)
for b in bars: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f'{b.get_height():.1f}', ha='center', fontsize=8)

ax = axes[2]
bars = ax.bar(short, mb_v, color=colors, alpha=0.85)
ax.set_title('Размер модели (МБ)')
ax.set_xticklabels(short, rotation=20, ha='right', fontsize=8); ax.grid(axis='y', alpha=0.3)
for b in bars: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f'{b.get_height():.0f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('./runs/results/comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()

# Scatter: скорость vs качество
fig2, ax2 = plt.subplots(figsize=(9,5))
for i,r in enumerate(ALL_METRICS):
    ax2.scatter(r['мс/кадр'], r['mAP@0.5'], s=r['МБ']*3, color=colors[i], alpha=0.8, label=r['Модель'], edgecolors='black', linewidths=0.5)
    ax2.annotate(short[i], (r['мс/кадр'], r['mAP@0.5']), textcoords='offset points', xytext=(8,4), fontsize=9)
ax2.set_xlabel('Время инференса мс/кадр (меньше = лучше)')
ax2.set_ylabel('mAP@0.5 (больше = лучше)')
ax2.set_title('Качество vs Скорость (размер точки = размер модели)')
ax2.legend(fontsize=8, bbox_to_anchor=(1.05,1), loc='upper left'); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('./runs/results/speed_vs_quality.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Лучшая модель и обоснование

In [ ]:
best = max(ALL_METRICS, key=lambda r: r['mAP@0.5'])
print('=' * 55)
print('ЛУЧШАЯ МОДЕЛЬ ПО mAP@0.5:')
print(f"  {best['Модель']}  ({best['Семейство']})")
print(f"  mAP@0.5:   {best['mAP@0.5']}")
print(f"  Precision: {best['Precision']}")
print(f"  Recall:    {best['Recall']}")
print(f"  Инференс:  {best['мс/кадр']} мс/кадр")
print(f"  Размер:    {best['МБ']} МБ")
print(f"  Примечание: {best['notes']}")
print('=' * 55)

---
## Демо: загрузи своё фото полки

In [ ]:
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont
from google.colab import files
import io

HISTORY_PATH = './runs/history.json'
def load_history(): return json.load(open(HISTORY_PATH)) if Path(HISTORY_PATH).exists() else []
def save_history(e):
    h = load_history(); h.append(e)
    json.dump(h, open(HISTORY_PATH,'w'), indent=2, ensure_ascii=False)

def draw_boxes(img_array, boxes, scores):
    img = Image.fromarray(img_array).convert('RGB')
    draw = ImageDraw.Draw(img)
    try: font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 14)
    except: font = ImageFont.load_default()
    for box, score in zip(boxes, scores):
        x1,y1,x2,y2 = [int(v) for v in box]
        color = '#00C853' if score > 0.7 else '#FFD600'
        draw.rectangle([x1,y1,x2,y2], outline=color, width=2)
        draw.text((x1+2,y1+2), f'{score:.2f}', fill=color, font=font)
    draw.rectangle([8,8,180,36], fill=(0,0,0))
    draw.text((10,10), f'Товаров: {len(boxes)}', fill='white', font=font)
    return img

CONF = 0.5
DEMO_WEIGHTS = best['weights']
DEMO_MODEL   = best['Модель']
print(f'Используется: {DEMO_MODEL}')
print('Загрузи фото полки магазина:')
uploaded = files.upload()

for fname, data in uploaded.items():
    img = Image.open(io.BytesIO(data)).convert('RGB')
    img_np = np.array(img)
    if 'DETR' in DEMO_MODEL:
        m = RTDETR(DEMO_WEIGHTS)
    elif 'YOLO' in DEMO_MODEL:
        m = YOLO(DEMO_WEIGHTS)
    else:
        m = None
    if m is not None:
        t0 = time.time()
        res = m.predict(img_np, conf=CONF, verbose=False)
        ms  = round((time.time()-t0)*1000, 1)
        boxes  = res[0].boxes.xyxy.cpu().numpy().tolist() if res[0].boxes else []
        scores = res[0].boxes.conf.cpu().numpy().tolist() if res[0].boxes else []
    else:
        dm = frcnn if 'Faster' in DEMO_MODEL else ssd_model
        dm.eval()
        tfm = T.Compose([T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
        inp = [tfm(img).to(device)]
        t0 = time.time()
        with torch.no_grad(): out = dm(inp)[0]
        ms = round((time.time()-t0)*1000, 1)
        keep = out['scores'] > CONF
        boxes  = out['boxes'][keep].cpu().numpy().tolist()
        scores = out['scores'][keep].cpu().numpy().tolist()
    count = len(boxes)
    avg_c = round(sum(scores)/len(scores),3) if scores else 0
    display(draw_boxes(img_np, boxes, scores))
    print(f'Найдено товаров: {count} | Уверенность: {avg_c} | Инференс: {ms}мс')
    save_history({'timestamp': datetime.datetime.now().isoformat(), 'file': fname,
                  'model': DEMO_MODEL, 'count': count, 'avg_confidence': avg_c, 'inference_ms': ms})
    print('История сохранена ->', HISTORY_PATH)

---
## Сохранение в Google Drive

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/variant10_shelf_detection'
shutil.copytree('./runs/results', DRIVE+'/results', dirs_exist_ok=True)
shutil.copy(best['weights'], DRIVE+f"/best_{best['Модель'].replace(' ','_')}.pt")
shutil.copy(HISTORY_PATH, DRIVE+'/history.json')
print('Сохранено в Google Drive:', DRIVE)

---
## Скачать результаты на компьютер

In [ ]:
from google.colab import files
files.download('./runs/results/comparison.csv')
files.download('./runs/results/comparison_chart.png')
files.download('./runs/results/speed_vs_quality.png')
files.download('./runs/history.json')
print('Файлы скачаны')